# 06 — LSTM para S0 y S1

Ambas reutilizan `cargar_serie` / `partir_train_test` de `src/utils_ts.py` y
`construir_resultado` / `tabla_comparativa` de `src/evaluacion_modelos.py`,
con el mismo esquema `COLUMNAS_RESULTADOS` que ya usan ARIMA/Prophet/
Holt-Winters. Los campos que no aplican a LSTM (`AIC`, `BIC`, `Ljung_Box_p`,
`Jarque_Bera_p`) quedan en NaN.

## Sección S0

Entrena **2 configuraciones distintas de LSTM** para `S0_total`, cambiando
`lookback`, número de capas y `dropout` entre una y otra, con estrategia de
pronóstico recursiva (walk-forward): en ningún paso se usa el valor real de
prueba, igual que los modelos clásicos del Laboratorio 1.

In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras

RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

from evaluacion_modelos import construir_resultado, tabla_comparativa
from utils_ts import cargar_serie, partir_train_test

DIR_RESULTADOS = RAIZ / "data" / "processed" / "resultados"
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
RUTA_CSV_AVANCE = DIR_RESULTADOS / "lstm_avance_s0_s1.csv"

SEED = 42

serie_s0 = cargar_serie("S0_total")
train_s0, test_s0 = partir_train_test(serie_s0)
print(f"S0_total -> train: {len(train_s0)} meses, test: {len(test_s0)} meses")

In [ ]:
def fijar_semillas(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def crear_ventanas(valores_escalados, lookback):
    """(X, y) con X.shape=(n, lookback, 1), y.shape=(n,)."""
    valores_escalados = np.asarray(valores_escalados, dtype=float).reshape(-1)
    X, y = [], []
    for i in range(lookback, len(valores_escalados)):
        X.append(valores_escalados[i - lookback:i])
        y.append(valores_escalados[i])
    X = np.array(X).reshape(-1, lookback, 1)
    y = np.array(y)
    return X, y


def construir_modelo(lookback, unidades, capas=1, dropout=0.0):
    """Devuelve un keras.Sequential: `capas` bloques LSTM (return_sequences=True
    salvo el último), dropout tras cada bloque si dropout>0, capa Dense(1) final.
    compile(optimizer="adam", loss="mse")."""
    modelo = keras.Sequential()
    modelo.add(keras.layers.Input(shape=(lookback, 1)))
    for indice_capa in range(capas):
        es_ultima = indice_capa == capas - 1
        modelo.add(keras.layers.LSTM(unidades, return_sequences=not es_ultima))
        if dropout > 0:
            modelo.add(keras.layers.Dropout(dropout))
    modelo.add(keras.layers.Dense(1))
    modelo.compile(optimizer="adam", loss="mse")
    return modelo


def pronosticar_recursivo(modelo, escalador, ultima_ventana, pasos):
    """Pronóstico walk-forward: predice un paso, reinyecta la predicción como
    último valor de la ventana y repite hasta completar `pasos`. Nunca usa
    valores reales de test. Devuelve el arreglo ya des-escalado."""
    ventana = np.asarray(ultima_ventana, dtype=float).reshape(-1).copy()
    predicciones = []
    for _ in range(pasos):
        entrada = ventana.reshape(1, len(ventana), 1)
        pred_escalada = modelo.predict(entrada, verbose=0)[0, 0]
        predicciones.append(pred_escalada)
        ventana = np.append(ventana[1:], pred_escalada)
    predicciones = np.array(predicciones).reshape(-1, 1)
    return escalador.inverse_transform(predicciones).reshape(-1)


def entrenar_lstm(train, test, lookback, unidades, capas=1, dropout=0.0,
                   epochs=100, batch_size=16, seed=SEED):
    """Escala train con MinMaxScaler (fit solo en train), entrena con
    EarlyStopping(monitor="loss", patience=10, restore_best_weights=True),
    pronostica los len(test) meses con pronosticar_recursivo.
    Devuelve {"modelo", "historial", "escalador",
              "y_pred": pd.Series alineada a test.index}."""
    fijar_semillas(seed)
    escalador = MinMaxScaler(feature_range=(0, 1))
    train_escalado = escalador.fit_transform(
        train.to_numpy().reshape(-1, 1)
    ).reshape(-1)

    X, y = crear_ventanas(train_escalado, lookback)
    modelo = construir_modelo(lookback, unidades, capas=capas, dropout=dropout)

    detencion_temprana = keras.callbacks.EarlyStopping(
        monitor="loss", patience=10, restore_best_weights=True
    )
    historial = modelo.fit(
        X, y,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[detencion_temprana],
        verbose=0,
    )

    ultima_ventana = train_escalado[-lookback:]
    y_pred = pronosticar_recursivo(modelo, escalador, ultima_ventana, len(test))
    y_pred = pd.Series(y_pred, index=test.index, name="y_pred")

    return {
        "modelo": modelo,
        "historial": historial,
        "escalador": escalador,
        "y_pred": y_pred,
    }

### Dos configuraciones de partida, con al menos un hiperparámetro distinto

- **Config 1** (`LSTM-1capa-w12`): `lookback=12, unidades=50, capas=1,
  dropout=0.0` — una LSTM simple de una capa y ventana corta (un año).
- **Config 2** (`LSTM-2capas-w24`): `lookback=24, unidades=64, capas=2,
  dropout=0.2` — más profunda, con ventana más larga (dos años) y
  regularización por dropout.

Fijos en ambas: `epochs=100, batch_size=16, seed=SEED`. Los hiperparámetros
que cambian entre configuraciones son `lookback`, `capas` y `dropout`.

In [ ]:
configuraciones_s0 = {
    "LSTM-1capa-w12": dict(lookback=12, unidades=50, capas=1, dropout=0.0),
    "LSTM-2capas-w24": dict(lookback=24, unidades=64, capas=2, dropout=0.2),
}

resultados_s0 = {}
filas_s0 = []
for nombre_modelo, parametros in configuraciones_s0.items():
    print(f"Entrenando {nombre_modelo}: {parametros}")
    resultado = entrenar_lstm(
        train_s0, test_s0, **parametros, epochs=100, batch_size=16, seed=SEED
    )
    resultados_s0[nombre_modelo] = resultado

    fila = construir_resultado(
        serie="S0_total",
        modelo=nombre_modelo,
        y_real=test_s0,
        y_pred=resultado["y_pred"],
        parametros={**parametros, "epochs": 100, "batch_size": 16},
        transformacion="MinMax(0,1)",
    )
    filas_s0.append(fila)
    print(f"  MAE={fila['MAE']:.2f}  RMSE={fila['RMSE']:.2f}  "
          f"MAPE={fila['MAPE']:.2f}%  "
          f"(épocas corridas: {len(resultado['historial'].history['loss'])})")

### Tabla comparativa y CSV del avance

Se guarda en `data/processed/resultados/lstm_avance_s0_s1.csv`.

In [ ]:
if RUTA_CSV_AVANCE.exists():
    previo = pd.read_csv(RUTA_CSV_AVANCE)
    previo = previo[previo["serie"] != "S0_total"]
    filas_combinadas = pd.concat(
        [previo, pd.DataFrame(filas_s0)], ignore_index=True
    ).to_dict("records")
else:
    filas_combinadas = filas_s0

tabla_avance = tabla_comparativa(filas_combinadas)
tabla_avance.to_csv(RUTA_CSV_AVANCE, index=False)
tabla_avance[tabla_avance["serie"] == "S0_total"]

### Real vs. pronóstico de prueba, ambas configuraciones de S0

Solo para verificar visualmente que el pronóstico recursivo es razonable;
la selección del mejor modelo y la comparación contra el Laboratorio 1
quedan para la entrega final.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(test_s0.index, test_s0.values, color="#20242b", linewidth=2.0,
        label="Observado")
colores = {"LSTM-1capa-w12": "#2f6690", "LSTM-2capas-w24": "#c44536"}
for nombre_modelo, resultado in resultados_s0.items():
    ax.plot(resultado["y_pred"].index, resultado["y_pred"].values,
            linewidth=1.4, color=colores[nombre_modelo], label=nombre_modelo)
ax.set_title("S0 — real vs. pronóstico LSTM (prueba, 63 meses)")
ax.set_ylabel("Viajeros")
ax.grid(alpha=0.25)
ax.legend(fontsize=9)
fig.tight_layout()
DIR_IMG = RAIZ / "informe" / "img"
DIR_IMG.mkdir(parents=True, exist_ok=True)
fig.savefig(DIR_IMG / "7_s0_lstm_avance.png", dpi=150, bbox_inches="tight")
plt.show()

### Cierre de la sección S0

- Se tunearon `lookback`, número de `capas` y `dropout` entre las dos
  configuraciones (ver tabla anterior para MAE/RMSE/MAPE exactos).
- El pronóstico es recursivo (walk-forward): en ningún paso se usa el valor
  real de prueba.
- El CSV queda guardado y el cuaderno es reproducible de principio a fin
  (semilla fija `SEED=42`).